# Day 2: 从零构建 GPT — 字符级语言模型

**目标**：从零实现一个完整的 GPT 模型，理解自回归生成的核心机制。

今天我们将亲手搭建 GPT 的每一个组件：Embedding、Self-Attention、Causal Mask、Multi-Head Attention、TransformerBlock，最终用它来生成莎士比亚风格的文本。

## 从分类到生成：BERT vs GPT

Day 1 我们做了**分类**（BERT 式），今天转向**生成**（GPT 式）。两者的核心区别：

| 特性 | BERT（Encoder） | GPT（Decoder） |
|------|-----------------|----------------|
| 注意力方向 | **双向**：每个 token 看所有 token | **单向**：每个 token 只看前面的 |
| 训练目标 | 完形填空（Masked LM） | 预测下一个 token |
| 典型应用 | 分类、NER、语义理解 | 文本生成、对话、代码生成 |
| Causal Mask | 不需要 | **必须**（防止偷看未来） |

ChatGPT、Claude 等大语言模型都是 GPT 架构（Decoder-only Transformer）。

## 为什么用字符级 Tokenizer？

实际的 LLM 使用 BPE/SentencePiece 等子词分词器，但字符级分词器：
- 词表极小（只有 65 个字符），模型很轻量
- **无需任何外部依赖**，纯 PyTorch 实现
- 概念完全相同，只是粒度不同

理解了字符级 GPT，你就理解了 GPT-4 的核心架构。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {DEVICE}')

---
# Part 1: 从零构建 GPT

## 1.1 字符级 Tokenizer

In [ ]:
# 加载 Shakespeare 数据集
from datasets import load_dataset

print('加载 tiny_shakespeare 数据集...')
ds = load_dataset('tiny_shakespeare', trust_remote_code=True)
text = ds['train']['text'][0]
print(f'数据集总字符数: {len(text):,}')
print(f'\n前 300 个字符:\n{text[:300]}')

In [ ]:
# 构建字符级 Tokenizer
# 收集所有出现过的字符作为词表
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)
print(f'词表大小: {VOCAB_SIZE}')
print(f'所有字符: {repr("".join(chars))}')

# 字符 <-> 索引 的映射表
stoi = {ch: i for i, ch in enumerate(chars)}  # string to index
itos = {i: ch for i, ch in enumerate(chars)}  # index to string

def encode(s):
    """将字符串编码为整数列表"""
    return [stoi[c] for c in s]

def decode(ids):
    """将整数列表解码为字符串"""
    return ''.join([itos[i] for i in ids])

In [ ]:
# 演示 encode / decode
test_str = 'Hello, World!'
encoded = encode(test_str)
decoded = decode(encoded)

print(f'原文:   "{test_str}"')
print(f'编码后: {encoded}')
print(f'解码回: "{decoded}"')
print(f'\n与 BPE 的对比：BPE 可能把 "Hello" 编码成 1 个 token，')
print(f'而字符级编码成了 {len(encode("Hello"))} 个 token')

In [ ]:
# 编码整个数据集
data = torch.tensor(encode(text), dtype=torch.long)
print(f'编码后 tensor 形状: {data.shape}')
print(f'前 20 个 token: {data[:20].tolist()}')
print(f'对应字符: "{decode(data[:20].tolist())}"')

# 划分训练集 / 验证集 (90% / 10%)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f'\n训练集: {len(train_data):,} tokens')
print(f'验证集: {len(val_data):,} tokens')

## 1.2 Embedding 可视化

Embedding 层将离散的 token ID 映射为连续的向量。训练过程中，语义相近的 token 会在向量空间中靠近。

In [ ]:
# Embedding 基础演示
d_model = 16  # 先用小维度演示
emb = nn.Embedding(VOCAB_SIZE, d_model)

# 查看几个字符的嵌入向量
sample_chars = ['a', 'b', 'A', 'B', ' ', '\n']
sample_ids = torch.tensor([stoi[c] for c in sample_chars])
sample_embs = emb(sample_ids)

print('字符嵌入向量（初始化是随机的）:')
for i, ch in enumerate(sample_chars):
    print(f'  {repr(ch):4s} (id={stoi[ch]:2d}): {sample_embs[i, :6].tolist()}...')

In [ ]:
# 可视化：用随机初始化的 embedding 计算字符间的余弦相似度
with torch.no_grad():
    all_ids = torch.arange(VOCAB_SIZE)
    all_embs = emb(all_ids)  # (VOCAB_SIZE, d_model)
    # 归一化
    normed = all_embs / all_embs.norm(dim=1, keepdim=True)
    # 余弦相似度矩阵
    sim_matrix = normed @ normed.T  # (VOCAB_SIZE, VOCAB_SIZE)

# 只展示字母部分
letters = 'abcdefghijABCDEFGHIJ'
letter_ids = [stoi[c] for c in letters]
sub_sim = sim_matrix[letter_ids][:, letter_ids]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sub_sim.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(letters)))
ax.set_yticks(range(len(letters)))
ax.set_xticklabels(list(letters), fontsize=9)
ax.set_yticklabels(list(letters), fontsize=9)
ax.set_title('字符 Embedding 余弦相似度（随机初始化）')
plt.colorbar(im)
plt.tight_layout()
plt.show()
print('现在是随机的，训练后相近字符（如大小写对）会变得更相似。')

## 1.3 Self-Attention：Q/K/V 详解

Self-Attention 的核心思想：每个 token 通过 **Query**（我在找什么）、**Key**（我能提供什么）、**Value**（我的实际内容）三个投影来交换信息。

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
# 手写 Single Head Self-Attention
class SingleHeadAttention(nn.Module):
    """单头自注意力 — 最基础的 attention 实现"""
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # Query 投影
        self.W_k = nn.Linear(d_model, d_model, bias=False)  # Key 投影
        self.W_v = nn.Linear(d_model, d_model, bias=False)  # Value 投影
        self.d_model = d_model
    
    def forward(self, x, use_causal_mask=False):
        B, T, C = x.shape  # batch, seq_len, d_model
        
        # 1. 计算 Q, K, V
        Q = self.W_q(x)  # (B, T, C) — 每个 token 的"查询向量"
        K = self.W_k(x)  # (B, T, C) — 每个 token 的"键向量"
        V = self.W_v(x)  # (B, T, C) — 每个 token 的"值向量"
        
        # 2. 计算注意力分数: QK^T / sqrt(d_k)
        scale = math.sqrt(C)
        attn_scores = (Q @ K.transpose(-2, -1)) / scale  # (B, T, T)
        
        # 3. (可选) 应用 causal mask
        if use_causal_mask:
            mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        
        # 4. Softmax → 注意力权重
        attn_weights = F.softmax(attn_scores, dim=-1)  # (B, T, T)
        
        # 5. 加权求和
        output = attn_weights @ V  # (B, T, C)
        
        return output, attn_weights

# 测试
sha = SingleHeadAttention(d_model=8)
x_test = torch.randn(1, 4, 8)  # 1 个样本, 4 个 token, 8 维
out, weights = sha(x_test, use_causal_mask=False)
print(f'输入形状:  {x_test.shape}')
print(f'输出形状:  {out.shape}')
print(f'注意力权重形状: {weights.shape}')
print(f'\n注意力权重（无 causal mask）:')
print(weights[0].detach())

## 1.4 Causal Mask — 为什么 GPT 只看前面？

GPT 的训练目标是**预测下一个 token**。如果位置 `i` 能看到位置 `i+1` 的 token，那预测就变成了"抄答案"而不是学习。

**Causal Mask（因果掩码）** 就是一个上三角矩阵，把"未来"的位置设为 `-inf`，softmax 后变成 0。

In [ ]:
# 可视化 Causal Mask
T = 6  # 序列长度

# 上三角矩阵: True 表示需要被 mask 掉的位置
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Mask 本身
axes[0].imshow(causal_mask.float().numpy(), cmap='Reds', vmin=0, vmax=1)
axes[0].set_title('Causal Mask\n(红色 = 被屏蔽)')
axes[0].set_xlabel('Key 位置')
axes[0].set_ylabel('Query 位置')
for i in range(T):
    for j in range(T):
        axes[0].text(j, i, 'X' if causal_mask[i,j] else '', ha='center', va='center', fontsize=10)

# 2. Mask 前的注意力分数（随机）
torch.manual_seed(42)
raw_scores = torch.randn(T, T)
axes[1].imshow(raw_scores.numpy(), cmap='coolwarm')
axes[1].set_title('原始注意力分数')

# 3. Mask 后 softmax
masked_scores = raw_scores.masked_fill(causal_mask, float('-inf'))
attn_weights_demo = F.softmax(masked_scores, dim=-1)
im = axes[2].imshow(attn_weights_demo.numpy(), cmap='Blues', vmin=0)
axes[2].set_title('Mask + Softmax 后\n(每个位置只关注前面)')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

print('关键观察：第 i 行只有前 i+1 个位置有非零权重')
print('这保证了每个 token 只能利用"过去"的信息来预测"未来"')

In [ ]:
# 对比有无 causal mask 的注意力权重
out_no_mask, w_no_mask = sha(x_test, use_causal_mask=False)
out_masked, w_masked = sha(x_test, use_causal_mask=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(w_no_mask[0].detach().numpy(), cmap='Blues', vmin=0, vmax=0.5)
ax1.set_title('无 Causal Mask（双向注意力）\nBERT 风格')
ax1.set_xlabel('Key')
ax1.set_ylabel('Query')
for i in range(4):
    for j in range(4):
        ax1.text(j, i, f'{w_no_mask[0,i,j]:.2f}', ha='center', va='center', fontsize=9)

ax2.imshow(w_masked[0].detach().numpy(), cmap='Blues', vmin=0, vmax=0.8)
ax2.set_title('有 Causal Mask（单向注意力）\nGPT 风格')
ax2.set_xlabel('Key')
ax2.set_ylabel('Query')
for i in range(4):
    for j in range(4):
        ax2.text(j, i, f'{w_masked[0,i,j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print('右图中上三角全为 0 — GPT 无法"偷看"未来的 token')

## 1.5 Multi-Head Attention

单头注意力只能学习一种"关注模式"。**多头注意力**将 Q, K, V 分成多个头，每个头独立学习不同的关注模式，最后拼接。

比如：一个头关注语法关系，另一个头关注语义关系。

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    多头自注意力（Multi-Head Self-Attention）
    
    将 d_model 拆成 n_head 个头，每个头的维度 = d_model / n_head
    每个头独立计算 attention，最后拼接回 d_model 维
    """
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0, 'd_model 必须能被 n_head 整除'
        self.n_head = n_head
        self.d_head = d_model // n_head  # 每个头的维度
        
        # Q, K, V 投影（合并成一个矩阵，提高效率）
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        
        # 1. 计算 Q, K, V
        qkv = self.qkv_proj(x)             # (B, T, 3*C)
        q, k, v = qkv.chunk(3, dim=-1)     # 各 (B, T, C)
        
        # 2. 拆成多头: (B, T, C) → (B, n_head, T, d_head)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        
        # 3. 计算注意力: QK^T / sqrt(d_head)
        scale = math.sqrt(self.d_head)
        attn = (q @ k.transpose(-2, -1)) / scale  # (B, n_head, T, T)
        
        # 4. Causal Mask
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        attn = attn.masked_fill(causal_mask, float('-inf'))
        
        # 5. Softmax + Dropout
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        
        # 6. 加权求和 + 合并多头
        out = (attn @ v)                                    # (B, n_head, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        
        # 7. 输出投影
        out = self.resid_dropout(self.out_proj(out))
        return out

# 测试
mha = MultiHeadAttention(d_model=32, n_head=4)
x_test = torch.randn(2, 8, 32)  # 2个样本, 8个token, 32维
out = mha(x_test)
print(f'输入: {x_test.shape} → 输出: {out.shape}')
print(f'4 个头, 每个头维度: {32 // 4} = 8')

## 1.6 TransformerBlock: LayerNorm + 残差连接

一个 TransformerBlock 由两个子层组成：
1. **Multi-Head Attention** + 残差连接
2. **Feed-Forward Network (FFN)** + 残差连接

每个子层前都有 **LayerNorm**（Pre-LN 架构，训练更稳定）。

```
x → LayerNorm → MHA → + (残差) → LayerNorm → FFN → + (残差) → 输出
```

In [ ]:
class FeedForward(nn.Module):
    """
    前馈网络：两层 MLP，隐藏层 4 倍扩展
    d_model → 4*d_model → d_model
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),  # 扩展
            nn.GELU(),                        # GELU 激活（比 ReLU 更平滑）
            nn.Linear(4 * d_model, d_model),  # 压缩回来
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """
    Transformer Block = Pre-LN + MHA + 残差 + Pre-LN + FFN + 残差
    
    残差连接的作用：让梯度能直接流回浅层，防止梯度消失
    LayerNorm 的作用：稳定训练，加速收敛
    """
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)          # 第一个 LayerNorm
        self.attn = MultiHeadAttention(d_model, n_head, dropout)  # 多头注意力
        self.ln2 = nn.LayerNorm(d_model)          # 第二个 LayerNorm
        self.ffn = FeedForward(d_model, dropout)  # 前馈网络
    
    def forward(self, x):
        # 子层 1: LayerNorm → MHA → 残差连接
        x = x + self.attn(self.ln1(x))
        # 子层 2: LayerNorm → FFN → 残差连接
        x = x + self.ffn(self.ln2(x))
        return x

# 测试
block = TransformerBlock(d_model=32, n_head=4)
x_test = torch.randn(2, 8, 32)
out = block(x_test)
print(f'TransformerBlock: {x_test.shape} → {out.shape}')
print(f'形状不变！残差连接保证了输入输出维度一致。')

## 1.7 完整 GPT 模型 + generate 方法

In [ ]:
class GPT(nn.Module):
    """
    完整的 GPT 模型
    
    结构：
    1. Token Embedding:    token ID → d_model 维向量
    2. Position Embedding: 位置编码（可学习的）
    3. Dropout
    4. N 层 TransformerBlock
    5. LayerNorm
    6. LM Head:            d_model → vocab_size（预测下一个 token 的概率）
    
    权重共享：LM Head 和 Token Embedding 共享权重（减少参数，效果更好）
    """
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        
        # 嵌入层
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        
        # N 层 Transformer
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)
        ])
        
        # 输出层
        self.ln_f = nn.LayerNorm(d_model)       # 最终 LayerNorm
        self.lm_head = nn.Linear(d_model, vocab_size)  # 语言模型头
        
        # 权重共享: embedding 和 lm_head 用同一组权重
        self.lm_head.weight = self.token_emb.weight
        
        # 初始化权重
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """使用较小的标准差初始化，帮助训练稳定"""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        """
        idx:     (B, T) 输入 token 索引
        targets: (B, T) 目标 token 索引（训练时才需要）
        """
        B, T = idx.shape
        
        # 1. Token Embedding + Position Embedding
        tok_emb = self.token_emb(idx)                         # (B, T, d_model)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, d_model)
        x = self.drop(tok_emb + pos_emb)
        
        # 2. N 层 Transformer
        x = self.blocks(x)
        x = self.ln_f(x)
        
        # 3. 预测下一个 token 的概率分布
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        # 4. 计算 loss（如果提供了 targets）
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),  # (B*T, vocab_size)
                targets.view(-1)                    # (B*T,)
            )
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        自回归生成：逐个预测下一个 token
        
        temperature: 控制随机性
          - temperature=0.1: 几乎确定性，总是选概率最高的
          - temperature=1.0: 标准采样
          - temperature=2.0: 更随机，更有"创意"
        
        top_k: 只从概率最高的 k 个 token 中采样
        """
        for _ in range(max_new_tokens):
            # 截断到 block_size（只保留最近的上下文）
            idx_cond = idx[:, -self.block_size:]
            
            # Forward pass
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # 只取最后位置
            
            # Top-k 过滤
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # 采样
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        
        return idx

In [ ]:
# 模型配置
BLOCK_SIZE = 128   # 上下文窗口长度
D_MODEL = 128      # 嵌入维度
N_HEAD = 4         # 注意力头数
N_LAYER = 4        # Transformer 层数
DROPOUT = 0.1

gpt = GPT(VOCAB_SIZE, D_MODEL, N_HEAD, N_LAYER, BLOCK_SIZE, DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in gpt.parameters())
print(f'GPT 模型参数量: {total_params:,}')
print(f'\n模型结构:')
print(f'  词表大小: {VOCAB_SIZE}')
print(f'  嵌入维度: {D_MODEL}')
print(f'  注意力头数: {N_HEAD} (每头 {D_MODEL // N_HEAD} 维)')
print(f'  Transformer 层数: {N_LAYER}')
print(f'  上下文窗口: {BLOCK_SIZE}')

In [ ]:
# 训练前，模型生成的是纯随机的文本
prompt = 'ROMEO:'
input_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
output_ids = gpt.generate(input_ids, max_new_tokens=100, temperature=1.0)
print('=== 训练前的生成（纯随机） ===')
print(decode(output_ids[0].tolist()))
print('\n可以看到完全是乱码 — 因为模型还没学到任何东西。')

---
# Part 2: 训练 Shakespeare GPT

现在让我们把数据准备好，开始训练！

In [ ]:
# 构建 Dataset 和 DataLoader
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    """
    字符级数据集
    x: 连续 block_size 个字符
    y: 右移一位（预测下一个字符）
    
    例如 block_size=4, 文本 "Hello":
      x = [H, e, l, l]  →  y = [e, l, l, o]
    """
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    
    def __len__(self):
        return len(self.data) - self.block_size
    
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

BATCH_SIZE = 64
train_dataset = CharDataset(train_data, BLOCK_SIZE)
val_dataset = CharDataset(val_data, BLOCK_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'训练集样本数: {len(train_dataset):,}')
print(f'验证集样本数: {len(val_dataset):,}')
print(f'每个 batch: {BATCH_SIZE} 个序列, 每个序列 {BLOCK_SIZE} 个 token')

# 看一个样本
x_sample, y_sample = train_dataset[0]
print(f'\n样本 x: "{decode(x_sample[:30].tolist())}..."')
print(f'样本 y: "{decode(y_sample[:30].tolist())}..."')
print('y 就是 x 右移一位 — 预测下一个字符！')

In [ ]:
# 评估函数
@torch.no_grad()
def evaluate(model, loader, device):
    """计算验证集平均 loss"""
    model.eval()
    total_loss = 0
    count = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        count += 1
    return total_loss / count

In [ ]:
# 训练循环
optimizer = torch.optim.AdamW(gpt.parameters(), lr=3e-4)
EPOCHS = 5

train_losses = []
val_losses = []

print('=' * 60)
print('开始训练字符级 GPT')
print('=' * 60)

for epoch in range(EPOCHS):
    gpt.train()
    start_time = time.time()
    total_loss = 0
    count = 0
    
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        # 训练五步曲
        logits, loss = gpt(x, y)          # Forward + Loss
        optimizer.zero_grad()              # 清零梯度
        loss.backward()                    # Backward
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)  # 梯度裁剪
        optimizer.step()                   # 参数更新
        
        total_loss += loss.item()
        count += 1
        
        if (batch_idx + 1) % 100 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {total_loss/count:.4f}')
    
    train_loss = total_loss / count
    val_loss = evaluate(gpt, val_loader, DEVICE)
    elapsed = time.time() - start_time
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f'\nEpoch {epoch+1}/{EPOCHS} | 耗时: {elapsed:.0f}s')
    print(f'  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    
    # 每个 epoch 生成一个样本看看效果
    gpt.eval()
    prompt_ids = torch.tensor([encode('ROMEO:\n')], dtype=torch.long, device=DEVICE)
    gen_ids = gpt.generate(prompt_ids, max_new_tokens=100, temperature=0.8, top_k=40)
    print(f'  生成样本: {decode(gen_ids[0].tolist())[:120]}...')
    print()

In [ ]:
# 训练曲线
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS+1), train_losses, marker='o', label='Train Loss')
ax.plot(range(1, EPOCHS+1), val_losses, marker='s', label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('GPT 训练曲线')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 生成文本样本 — 不同 temperature 对比
gpt.eval()

prompts = ['ROMEO:', 'To be or ', 'The king']
temperatures = [0.3, 0.8, 1.5]

for prompt in prompts:
    print(f'\n{"="*60}')
    print(f'Prompt: "{prompt}"')
    print('=' * 60)
    
    for temp in temperatures:
        input_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
        output_ids = gpt.generate(input_ids, max_new_tokens=150, temperature=temp, top_k=40)
        generated = decode(output_ids[0].tolist())
        print(f'\n--- temperature={temp} ---')
        print(generated[:200])

In [ ]:
# Temperature 的直觉理解
print('=== Temperature 对生成的影响 ===')
print()
print('Temperature 控制 softmax 的"锐度"：')
print('  - 低温 (0.1-0.3): 分布非常尖锐，几乎总选概率最高的 → 重复、保守')
print('  - 标准 (0.8-1.0): 平衡的随机性 → 连贯且有变化')
print('  - 高温 (1.5-2.0): 分布很平坦，随机性大 → 创意但可能不连贯')

# 可视化不同 temperature 下的概率分布
logits_demo = torch.tensor([2.0, 1.5, 0.5, 0.1, -0.5])
temps = [0.3, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
labels = ['A', 'B', 'C', 'D', 'E']

for ax, t in zip(axes, temps):
    probs = F.softmax(logits_demo / t, dim=0).numpy()
    ax.bar(labels, probs, color='steelblue')
    ax.set_title(f'Temperature = {t}')
    ax.set_ylim(0, 1)
    ax.set_ylabel('概率')
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 更多生成演示
gpt.eval()
print('=== 自由生成（temperature=0.8, top_k=40）===')
print()

# 用换行符作为起始
start = torch.tensor([encode('\n')], dtype=torch.long, device=DEVICE)
output = gpt.generate(start, max_new_tokens=500, temperature=0.8, top_k=40)
print(decode(output[0].tolist()))

---
# Part 3: 总结

## 今天从零构建了什么

```
字符级 Tokenizer
    ↓
Token Embedding + Position Embedding
    ↓
Multi-Head Self-Attention (with Causal Mask)
    ↓
TransformerBlock (LayerNorm + MHA + FFN + 残差连接)
    ↓
GPT 完整模型 + generate() 方法
    ↓
训练 + 生成莎士比亚文本
```

## 关键概念

| 组件 | 作用 | 在 GPT-4 / Claude 中 |
|------|------|--------------------|
| Causal Mask | 防止偷看未来 | 完全相同 |
| Multi-Head Attention | 多角度关注 | 更多头（如 96 头）|
| 权重共享 | 减少参数 | 同样使用 |
| Temperature | 控制生成随机性 | API 参数可调 |
| Top-k | 限制采样范围 | Top-p 更常用 |

## 思考题

1. **为什么 GPT 用 Causal Mask 而不是像 BERT 一样看全部上下文？** 提示：想想训练目标和生成过程的一致性。

2. **权重共享**（`lm_head.weight = token_emb.weight`）为什么有效？这在直觉上意味着什么？

3. **如果把 `block_size` 从 128 增加到 1024，模型能力会怎样变化？成本呢？** 提示：Self-attention 的计算复杂度是 O(T^2)。

4. **挑战题**：尝试加入学习率 warmup 和 cosine decay，观察对训练的影响。